# ChibiCreate — BENCHMARK COMPARATIVO: WAI-illustrious-SDXL

> ## BENCHMARK COMPARATIVO — NAO E PIPELINE OFICIAL
>
> Responde **uma** pergunta: *o WAI-illustrious-SDXL preserva a roupa e o
> design original melhor que o FLUX.2 klein 4B?*
>
> Nao altera o benchmark FLUX, as Runs 001/002/003 do FLUX, o Flow 01, os
> quality gates nem o design transfer.

---

## Multi-referencia via IP-Adapter

O checkpoint SDXL nao tem mecanismo proprio de referencia como o
`ReferenceLatent` do FLUX. Isso **nao** quer dizer que ele nao consiga usar
varias referencias: o **IP-Adapter** fornece multi-referencia real.

```
full_body ──► IPAdapterEncoder (peso 1.0) ──┐
face      ──► IPAdapterEncoder (peso 0.6) ──┼──► IPAdapterCombineEmbeds
outfit    ──► IPAdapterEncoder (peso 0.8) ──┘             │
                                                  IPAdapterEmbeds
                                                          │
                                                      KSampler
```

Encoder + Combine em vez de `IPAdapterAdvanced` empilhado em serie, porque
so assim o peso de **cada** referencia fica explicito e auditavel.

| run | referencias | workflow |
|---|---|---|
| 001 | `full_body` | `v1` |
| 002 | `full_body` — repeticao exata da 001 | `v1` |
| 003 | `full_body` + `face` + `outfit` | `v2` |

### Sobre a comparacao com o FLUX

- **FLUX** = multi-referencia pelo mecanismo proprio (`ReferenceLatent`)
- **WAI** = multi-referencia por **IP-Adapter**

Sao implementacoes **diferentes**. A comparacao e sobre o **resultado visual
com as mesmas referencias de entrada**, nunca sobre equivalencia de
arquitetura.

### Dependencia de terceiro

O IP-Adapter exige o custom node `cubiq/ComfyUI_IPAdapter_plus` e dois pesos
auxiliares. A celula 6 pede **aceite explicito** antes de instalar. Se os
nodes nao aparecerem no servidor, o benchmark **PARA** — a Run 003 nunca cai
para uma referencia em silencio.


In [ ]:
#@title 0. Personagem, run e pesos { display-mode: "form" }
#@markdown Unico ponto a configurar. As celulas de execucao nao devem ser editadas.
CHARACTER_ID = "waifu_001"  #@param {type:"string"}

#@markdown ---
#@markdown **Qual run executar.** 001 e 002 usam so `full_body` (a 002 e
#@markdown repeticao exata da 001). A 003 usa as tres referencias.
RUN = "WAI - Run 003 (full_body + face + outfit)"  #@param ["WAI - Run 001 (full_body)", "WAI - Run 002 (repeticao exata da 001)", "WAI - Run 003 (full_body + face + outfit)"]

#@markdown ---
#@markdown Reprodutibilidade: identicos nas tres runs.
SEED = 42  #@param {type:"integer"}

#@markdown ---
#@markdown **Pesos — BASELINE EXPERIMENTAL, nao validados.** Sem sweep nesta
#@markdown rodada: uma configuracao so, com os tres pesos registrados.
WEIGHT_FULL_BODY = 1.0  #@param {type:"slider", min:0.0, max:2.0, step:0.05}
WEIGHT_FACE = 0.6  #@param {type:"slider", min:0.0, max:2.0, step:0.05}
WEIGHT_OUTFIT = 0.8  #@param {type:"slider", min:0.0, max:2.0, step:0.05}
COMBINE_METHOD = "concat"  #@param ["concat", "add", "average", "norm average", "subtract", "max", "min"]

RUN_ID = RUN.split("Run ")[1][:3]
IS_RUN_003 = RUN_ID == "003"
MODEL_KEY = "wai_illustrious_sdxl_v170"
WORKFLOW = "experimental/wai_illustrious_ipadapter"
WORKFLOW_VERSION = "v2" if IS_RUN_003 else "v1"
REFS_DESTA_RUN = (["full_body", "face", "outfit"] if IS_RUN_003
                  else ["full_body"])

print("personagem  :", CHARACTER_ID)
print("run         :", RUN_ID)
print("workflow    :", f"{WORKFLOW}@{WORKFLOW_VERSION}")
print("referencias :", REFS_DESTA_RUN)
print("seed        :", SEED)
if IS_RUN_003:
    print("pesos       : full_body", WEIGHT_FULL_BODY,
          "| face", WEIGHT_FACE, "| outfit", WEIGHT_OUTFIT)
    print("combine     :", COMBINE_METHOD)
else:
    print("peso        : full_body", WEIGHT_FULL_BODY, "(sem combine)")
print()
print("Pesos sao BASELINE EXPERIMENTAL, nao valores validados.")


---

## Celula 1 — ambiente e repositorio

In [ ]:
#@title 1. Ambiente e repositorio { display-mode: "form" }
REPO_URL = "https://github.com/BloomRX/ChibiCreate"  #@param {type:"string"}
BRANCH   = "arena/01a07ece-chibicreate"  #@param {type:"string"}
ATUALIZAR_REPO = True  #@param {type:"boolean"}

import os, sys, subprocess, pathlib

%cd /content
if not pathlib.Path("/content/ChibiCreate/.git").exists():
    !git clone --branch $BRANCH $REPO_URL ChibiCreate
elif ATUALIZAR_REPO:
    !cd /content/ChibiCreate && git fetch origin $BRANCH && git checkout -B $BRANCH origin/$BRANCH

%cd /content/ChibiCreate
!git log --oneline -1

for _m in [k for k in list(sys.modules)
           if k.startswith("chibi") or k.startswith("scripts.chibi")]:
    del sys.modules[_m]
sys.path.insert(0, "/content/ChibiCreate")
sys.path.insert(0, "/content/ChibiCreate/scripts")

!pip install -q pyyaml pillow numpy


---

## Celula 2 — preflight (portao)

Se nao couber, **BLOCKED**. Sem fallback silencioso, sem CPU, sem trocar de
modelo, sem quantizar por conta propria.

In [ ]:
#@title 2. Preflight — GPU, VRAM, RAM, disco { display-mode: "form" }
import sys, json, shutil, subprocess, time

# Checkpoint SDXL fp16 (~8-10 GB) + IP-Adapter (~1 GB) + CLIP-Vision ViT-H
# (~2.5 GB no encode). Estimativa de registry, nao medicao nossa.
MIN_VRAM_GB = 12.0
MIN_DISK_GB = 20.0
MIN_RAM_GB  = 10.0

try:
    import torch
except ImportError:
    torch = None

print("=" * 62)
print("PREFLIGHT — detectado, nao presumido")
print("=" * 62)
print("Python :", sys.version.split()[0])
print("Torch  :", torch.__version__ if torch else "ausente")

if torch is None or not torch.cuda.is_available():
    print("CUDA   : INDISPONIVEL")
    raise SystemExit("BLOCKED — nenhuma GPU CUDA. Runtime -> GPU.")

props = torch.cuda.get_device_properties(0)
vram = props.total_memory / 1024 ** 3
free_disk = shutil.disk_usage("/content").free / 1024 ** 3
try:
    import psutil
    ram = psutil.virtual_memory().total / 1024 ** 3
except ImportError:
    ram = float(subprocess.check_output(
        ["awk", "/MemTotal/ {print $2/1048576}", "/proc/meminfo"]).strip())

GPU_INFO = {
    "name": props.name, "vram_total_gb": round(vram, 2),
    "vram_free_gb": round(torch.cuda.mem_get_info()[0] / 1024 ** 3, 2),
    "cuda": torch.version.cuda,
    "capability": f"{props.major}.{props.minor}",
    "torch": torch.__version__, "python": sys.version.split()[0],
    "bf16_supported": props.major >= 8,
    "ram_gb": round(ram, 2), "disk_free_gb": round(free_disk, 2),
}
for k, v in GPU_INFO.items():
    print(f"  {k:18} {v}")

print()
falhas = []
if vram < MIN_VRAM_GB:
    falhas.append(f"VRAM {vram:.1f} < {MIN_VRAM_GB} GB")
if free_disk < MIN_DISK_GB:
    falhas.append(f"disco {free_disk:.1f} < {MIN_DISK_GB} GB")
if ram < MIN_RAM_GB:
    falhas.append(f"RAM {ram:.1f} < {MIN_RAM_GB} GB")

if not GPU_INFO["bf16_supported"]:
    print("AVISO: sem bf16 nativo (capability < 8.0, ex. T4). Cai para fp16.")

if falhas:
    print("=" * 62); print("BLOCKED"); print("=" * 62)
    for f in falhas:
        print(" -", f)
    print()
    print("Nao fazer fallback silencioso: nao trocar de modelo, nao remover")
    print("referencias, nao quantizar por conta propria. Reportar o bloqueio.")
    raise SystemExit("BLOCKED")

print("Hardware adequado.")
json.dump(GPU_INFO, open("/content/gpu_info_wai.json", "w"), indent=2)


---

## Celula 3 — entradas e hashes

As **mesmas** referencias do benchmark FLUX, derivadas de
`characters/<CHARACTER_ID>/reference/`. Os originais nunca sao modificados.

In [ ]:
#@title 3. Referencias — full_body, face, outfit { display-mode: "form" }
import hashlib, pathlib, json
import numpy as np
from PIL import Image

REF_DIR = pathlib.Path(
    f"/content/ChibiCreate/characters/{CHARACTER_ID}/reference")
ARQUIVOS = {"full_body": "full_body.png", "face": "face.png",
            "outfit": "outfit.png"}

ENTRADAS = {}
for papel, arq in ARQUIVOS.items():
    p = REF_DIR / arq
    if not p.exists():
        print(f"  [ausente] {papel}: {p}")
        continue
    b = p.read_bytes()
    im = Image.open(p)
    ENTRADAS[papel] = {
        "file": arq, "path": str(p),
        "artifact_sha256": hashlib.sha256(b).hexdigest(),
        "pixel_sha256": hashlib.sha256(
            np.array(im.convert("RGBA")).tobytes()).hexdigest(),
        "size": list(im.size), "bytes": len(b),
        "usada_nesta_run": papel in REFS_DESTA_RUN,
    }
    marca = "USADA" if papel in REFS_DESTA_RUN else "nao usada nesta run"
    print(f"  {papel:10} {str(im.size):12} {ENTRADAS[papel]['artifact_sha256'][:16]}  [{marca}]")

faltando = [r for r in REFS_DESTA_RUN if r not in ENTRADAS]
if faltando:
    raise SystemExit(
        f"BLOCKED — a Run {RUN_ID} exige {REFS_DESTA_RUN} e faltam {faltando}. "
        "Nao executar com menos referencias em silencio.")

print()
print(f"Run {RUN_ID}: {len(REFS_DESTA_RUN)} referencia(s) confirmada(s).")
print("Arquivos NAO sao modificados — apenas lidos.")
json.dump(ENTRADAS, open("/content/wai_inputs.json", "w"), indent=2)


---

## Celula 4 — fixar a versao do checkpoint

O Civitai esta fora da allowlist de egress da sandbox onde este notebook foi
escrito, entao `modelVersionId`, arquivo e SHA256 estao `null` no registry —
nao foram inventados. Voce, no Colab, enxerga a pagina: preencha abaixo.

In [ ]:
#@title 4. Versao do checkpoint WAI { display-mode: "form" }
#@markdown Dados REAIS da pagina do Civitai. Nao inventar.
CIVITAI_MODEL_ID = "827184"  #@param {type:"string"}
CIVITAI_VERSION_ID = ""  #@param {type:"string"}
CKPT_FILENAME = ""  #@param {type:"string"}
CKPT_SHA256_ESPERADO = ""  #@param {type:"string"}
LICENCA_EXIBIDA = "Commercial use allowed (conforme UI do Civitai)"  #@param {type:"string"}

from chibi import model_registry as mr

CFG = mr.get_model(MODEL_KEY)
print("label       :", CFG["label"])
print("pipeline    :", CFG["pipeline_type"])
print("mecanismo   :", CFG["reference_mechanism"])
print("refs suport :", CFG["references_supported"])
print("licenca     :", CFG["license_name"], "| verified:", CFG["license_verified"])
print("status      :", CFG["status"])
print()
print("NAO AFIRMAR EQUIVALENCIA:")
print(" ", " ".join(CFG["reference_equivalence_note"].split()))
print()

VERSAO = {
    "civitai_model_id": CIVITAI_MODEL_ID,
    "civitai_model_version_id": CIVITAI_VERSION_ID or None,
    "file": CKPT_FILENAME or None,
    "sha256_expected": CKPT_SHA256_ESPERADO or None,
    "license_displayed": LICENCA_EXIBIDA,
    "source_of_record": "informado por humano no Colab (Civitai bloqueado na sandbox)",
    "verified_by_agent": False,
}
if not CIVITAI_VERSION_ID or not CKPT_FILENAME:
    print("[HUMAN REVIEW REQUIRED] Versao NAO fixada.")
    print("Sem modelVersionId + nome do arquivo o benchmark nao e reproduzivel.")
else:
    print("versao fixada:", CIVITAI_VERSION_ID, "|", CKPT_FILENAME)
json.dump(VERSAO, open("/content/wai_version.json", "w"), indent=2)


---

## Celula 5 — ComfyUI

In [ ]:
#@title 5. Subir o ComfyUI { display-mode: "form" }
import subprocess, time, urllib.request, json, pathlib, os

def comfy_no_ar(timeout=5):
    try:
        with urllib.request.urlopen(
                "http://127.0.0.1:8188/system_stats", timeout=timeout) as r:
            return json.load(r)
    except Exception:
        return None

if not pathlib.Path("/content/ComfyUI").exists():
    !git clone -q https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
    !pip install -q -r /content/ComfyUI/requirements.txt

COMFY_COMMIT = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd="/content/ComfyUI").decode().strip()
print("ComfyUI commit:", COMFY_COMMIT)

def subir_comfy():
    log = open("/content/comfyui_wai.log", "w")
    p = subprocess.Popen(
        ["python", "main.py", "--listen", "127.0.0.1", "--port", "8188"],
        cwd="/content/ComfyUI", stdout=log, stderr=subprocess.STDOUT)
    for i in range(120):
        time.sleep(5)
        if comfy_no_ar():
            print(f"  no ar apos ~{(i + 1) * 5}s")
            return p
        if p.poll() is not None:
            print(open("/content/comfyui_wai.log").read()[-3000:])
            raise SystemExit("ComfyUI morreu ao iniciar")
    raise SystemExit("ComfyUI nao respondeu em 10 min")

PROC = subir_comfy()
os.environ["CHIBI_COMFY_URL"] = "http://127.0.0.1:8188"


---

## Celula 6 — IP-Adapter: custom node + pesos

**Dependencia de terceiro, com aceite explicito.** Sem isto nao ha
multi-referencia — e a Run 003 **para** em vez de rodar com uma imagem so.

| item | arquivo | licenca |
|---|---|---|
| custom node | `cubiq/ComfyUI_IPAdapter_plus` | Apache-2.0 |
| adapter | `ip-adapter-plus_sdxl_vit-h.safetensors` | Apache-2.0 |
| encoder | `CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors` | Apache-2.0 |

A variante *plus* usa patch embeddings e fica mais proxima da referencia —
que e exatamente o eixo medido. O encoder ViT-H e o par obrigatorio dela;
trocar por bigG daria erro de dimensao de tensor.

In [ ]:
#@title 6. Instalar IP-Adapter (aceite explicito) { display-mode: "form" }
#@markdown Marque para instalar o custom node e baixar os dois pesos.
ACEITO_INSTALAR_IPADAPTER = False  #@param {type:"boolean"}

import pathlib, subprocess, hashlib, json, os

IPA_DIR = pathlib.Path("/content/ComfyUI/custom_nodes/ComfyUI_IPAdapter_plus")
MODELS = pathlib.Path("/content/ComfyUI/models")
CFG_IPA = CFG["ipadapter_models"]

def sha256_of(p, chunk=1 << 22):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

if not ACEITO_INSTALAR_IPADAPTER and not IPA_DIR.exists():
    print("=" * 62)
    print("BLOCKED — IP-Adapter nao instalado")
    print("=" * 62)
    print("custom node :", CFG["custom_node_repo"])
    print("nodes       :", ", ".join(CFG["custom_node_nodes"]))
    print()
    print("Sem ele nao ha multi-referencia. A Run 003 NAO roda com uma")
    print("referencia so — isso seria substituir multi-reference em silencio.")
    raise SystemExit("Marque ACEITO_INSTALAR_IPADAPTER para prosseguir.")

if not IPA_DIR.exists():
    !git clone -q {CFG["custom_node_repo"]} {IPA_DIR}

IPA_COMMIT = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=IPA_DIR).decode().strip()
IPA_TAG = subprocess.run(["git", "describe", "--tags", "--always"], cwd=IPA_DIR,
                         capture_output=True, text=True).stdout.strip()
print("IP-Adapter node:", CFG["custom_node_repo"])
print("  commit :", IPA_COMMIT)
print("  tag    :", IPA_TAG)

IPADAPTER_ASSETS = {}
for papel, spec in CFG_IPA.items():
    destino = MODELS / spec["target_dir"].split("/", 1)[1] / spec["file"]
    destino.parent.mkdir(parents=True, exist_ok=True)
    if not destino.exists():
        url = (f"https://huggingface.co/{spec['repo']}/resolve/main/"
               f"{spec['repo_path']}")
        print(f"  baixando {papel}: {spec['file']}")
        !wget -q --show-progress -O "{destino}" "{url}"
    real = sha256_of(destino)
    IPADAPTER_ASSETS[papel] = {
        "file": spec["file"], "repo": spec["repo"],
        "repo_path": spec["repo_path"], "license": spec["license"],
        "sha256": real, "size_bytes": destino.stat().st_size,
    }
    print(f"  {papel:11} {spec['file']}")
    print(f"    sha256   {real}")
    print(f"    tamanho  {destino.stat().st_size / 1e6:.0f} MB | {spec['license']}")

IPADAPTER_META = {
    "repo": CFG["custom_node_repo"],
    "commit": IPA_COMMIT, "revision": IPA_TAG,
    "nodes": CFG["custom_node_nodes"],
    "assets": IPADAPTER_ASSETS,
}
json.dump(IPADAPTER_META, open("/content/ipadapter_meta.json", "w"), indent=2)
print()
print("Reinicie o ComfyUI na proxima celula para carregar os nodes.")


In [ ]:
#@title 7. Reiniciar o ComfyUI e validar os nodes IP-Adapter { display-mode: "form" }
import subprocess, time, urllib.request, json

try:
    PROC.terminate(); PROC.wait(timeout=60)
except Exception:
    pass
time.sleep(3)
PROC = subir_comfy()

with urllib.request.urlopen(
        "http://127.0.0.1:8188/object_info", timeout=120) as r:
    OBJECT_INFO = json.load(r)

print("nodes no servidor:", len(OBJECT_INFO))
print()
faltando = [n for n in CFG["custom_node_nodes"] if n not in OBJECT_INFO]
for n in CFG["custom_node_nodes"]:
    print(f"  {'OK  ' if n in OBJECT_INFO else 'FALTA'} {n}")

if faltando:
    print()
    print("=" * 62)
    print("BLOCKED — nodes IP-Adapter ausentes")
    print("=" * 62)
    print("faltando:", faltando)
    print()
    print("Ver /content/comfyui_wai.log. Causa comum: ComfyUI desatualizado")
    print("(o IP-Adapter exige a versao mais recente).")
    print()
    print("NAO substituir multi-reference por uma imagem so. PARAR e reportar.")
    raise SystemExit("BLOCKED — IP-Adapter indisponivel")

print()
print("IP-Adapter disponivel. Multi-referencia possivel.")


---

## Celula 8 — validar o grafo

Confere o workflow da run escolhida contra os nodes reais do servidor. Se
falhar, **pare** — nao edite o grafo para "fazer passar".

In [ ]:
#@title 8. Validar o workflow desta run { display-mode: "form" }
import json, hashlib, pathlib

WF_PATH = pathlib.Path(
    f"/content/ChibiCreate/workflows/{WORKFLOW}/{WORKFLOW_VERSION}.json")
WF_BYTES = WF_PATH.read_bytes()
WORKFLOW_SHA256 = hashlib.sha256(WF_BYTES).hexdigest()
wf = json.loads(WF_BYTES)
nodes = {k: v for k, v in wf.items() if not k.startswith("_")}

print("workflow :", f"{WORKFLOW}@{WORKFLOW_VERSION}")
print("sha256   :", WORKFLOW_SHA256)
print("nodes    :", len(nodes))

classes = sorted({v["class_type"] for v in nodes.values()})
ausentes = [c for c in classes if c not in OBJECT_INFO]
print("classes  :", classes)
if ausentes:
    raise SystemExit(f"BLOCKED — classes ausentes no servidor: {ausentes}")

# O numero de referencias do grafo tem de bater com o da run escolhida.
n_load = sum(1 for v in nodes.values() if v["class_type"] == "LoadImage")
n_enc = sum(1 for v in nodes.values() if v["class_type"] == "IPAdapterEncoder")
print()
print("LoadImage        :", n_load)
print("IPAdapterEncoder :", n_enc)
assert n_load == len(REFS_DESTA_RUN) == n_enc, (
    f"Run {RUN_ID} pede {len(REFS_DESTA_RUN)} referencias mas o grafo tem "
    f"{n_load}. Nunca executar com contagem divergente.")

if IS_RUN_003:
    assert any(v["class_type"] == "IPAdapterCombineEmbeds"
               for v in nodes.values()), "Run 003 exige combine dos embeds"
    print("IPAdapterCombineEmbeds presente.")

print()
print(f"Grafo valido para a Run {RUN_ID} com {n_load} referencia(s).")


---

## Celula 9 — executar

Monta o grafo com os parametros e pesos declarados e envia ao ComfyUI.

In [ ]:
#@title 9. Executar a run { display-mode: "form" }
import json, time, copy, hashlib, pathlib, urllib.request, uuid
import numpy as np
from PIL import Image

if not VERSAO["civitai_model_version_id"] or not VERSAO["file"]:
    raise SystemExit(
        "BLOCKED — fixe a versao do checkpoint na celula 4 antes de executar.")

CKPT_DIR = pathlib.Path("/content/ComfyUI/models/checkpoints")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_PATH = CKPT_DIR / VERSAO["file"]
if not CKPT_PATH.exists():
    disponiveis = sorted(p.name for p in CKPT_DIR.glob("*.safetensors"))
    raise SystemExit(
        f"BLOCKED — checkpoint '{VERSAO['file']}' ausente. "
        f"Disponiveis: {disponiveis or 'nenhum'}. Baixe do Civitai (a API "
        "exige token) ou suba o arquivo para o diretorio acima.")

CKPT_SHA256 = sha256_of(CKPT_PATH)
if VERSAO.get("sha256_expected") and \
        CKPT_SHA256.lower() != VERSAO["sha256_expected"].lower():
    raise SystemExit("BLOCKED — SHA256 do checkpoint diverge do declarado.")
print("checkpoint:", CKPT_PATH.name)
print("sha256    :", CKPT_SHA256)

# Referencias vao para o input do ComfyUI (copia; originais intocados).
COMFY_INPUT = pathlib.Path("/content/ComfyUI/input")
COMFY_INPUT.mkdir(parents=True, exist_ok=True)
nomes = {}
for papel in REFS_DESTA_RUN:
    origem = pathlib.Path(ENTRADAS[papel]["path"])
    nome = f"{CHARACTER_ID}_{papel}.png"
    (COMFY_INPUT / nome).write_bytes(origem.read_bytes())
    nomes[papel] = nome

PAR = CFG["parameters"]
_reg = mr.load_registry()
PROMPT = " ".join(_reg["base_prompt"].split())
NEGATIVE = CFG["negative_prompt_override"]
PREFIX = f"wai_{CHARACTER_ID}_run{RUN_ID}_{uuid.uuid4().hex[:8]}"

SUBST = {
    "%%CKPT_NAME%%": VERSAO["file"],
    "%%PROMPT%%": PROMPT,
    "%%NEGATIVE_PROMPT%%": NEGATIVE,
    "%%IPADAPTER_FILE%%": CFG["ipadapter_models"]["adapter"]["file"],
    "%%CLIP_VISION_FILE%%": CFG["ipadapter_models"]["clip_vision"]["file"],
    "%%REF_FULL_BODY%%": nomes.get("full_body"),
    "%%REF_FACE%%": nomes.get("face"),
    "%%REF_OUTFIT%%": nomes.get("outfit"),
    "%%WEIGHT_FULL_BODY%%": float(WEIGHT_FULL_BODY),
    "%%WEIGHT_FACE%%": float(WEIGHT_FACE),
    "%%WEIGHT_OUTFIT%%": float(WEIGHT_OUTFIT),
    "%%COMBINE_METHOD%%": COMBINE_METHOD,
    "%%WEIGHT_TYPE%%": "linear",
    "%%EMBEDS_SCALING%%": "V only",
    "%%WIDTH%%": PAR["resolution"][0],
    "%%HEIGHT%%": PAR["resolution"][1],
    "%%SEED%%": int(SEED), "%%STEPS%%": PAR["steps"], "%%CFG%%": PAR["cfg"],
    "%%SAMPLER%%": PAR["sampler"], "%%SCHEDULER%%": PAR["scheduler"],
    "%%DENOISE%%": PAR["denoise"], "%%OUTPUT_PREFIX%%": PREFIX,
}

grafo = copy.deepcopy(nodes)
for nid, node in grafo.items():
    for campo, valor in node["inputs"].items():
        if isinstance(valor, str) and valor in SUBST:
            v = SUBST[valor]
            if v is None:
                raise SystemExit(f"BLOCKED — placeholder {valor} sem valor.")
            node["inputs"][campo] = v
    node.pop("_meta", None)

restantes = [v for n in grafo.values() for v in n["inputs"].values()
             if isinstance(v, str) and v.startswith("%%")]
assert not restantes, f"placeholders nao substituidos: {restantes}"

print()
print("PROMPT   :", PROMPT[:110], "...")
print("NEGATIVE :", NEGATIVE)
print("params   :", {k: SUBST[f"%%{k.upper()}%%"]
                     for k in ("seed", "steps", "cfg", "sampler", "scheduler")})
print("pesos    :", {p: SUBST[f"%%WEIGHT_{p.upper()}%%"] for p in REFS_DESTA_RUN})
print()

t0 = time.time()
req = urllib.request.Request(
    "http://127.0.0.1:8188/prompt",
    data=json.dumps({"prompt": grafo}).encode(),
    headers={"Content-Type": "application/json"})
try:
    with urllib.request.urlopen(req, timeout=60) as r:
        pid = json.load(r)["prompt_id"]
except urllib.error.HTTPError as e:
    print(e.read().decode()[:3000])
    raise SystemExit("BLOCKED — ComfyUI rejeitou o grafo. Erro acima.")

print("prompt_id:", pid)
hist = None
while True:
    time.sleep(3)
    with urllib.request.urlopen(
            f"http://127.0.0.1:8188/history/{pid}", timeout=30) as r:
        h = json.load(r)
    if pid in h:
        hist = h[pid]
        break
    if time.time() - t0 > 1800:
        raise SystemExit("timeout de 30 min")

EXEC_TIME = round(time.time() - t0, 2)
status = hist.get("status", {})
if status.get("status_str") == "error":
    print(json.dumps(status, indent=2)[:3000])
    raise SystemExit("BLOCKED — execucao falhou. Erro acima, sem mascarar.")

saidas = [i for o in hist["outputs"].values() for i in o.get("images", [])]
assert saidas, "nenhuma imagem produzida"
info = saidas[0]
OUT = (pathlib.Path("/content/ComfyUI/output") /
       (info.get("subfolder") or "") / info["filename"])
print(f"gerado em {EXEC_TIME}s -> {OUT.name}")
display(Image.open(OUT))


---

## Celula 10 — recipe e hashes

Registro completo de reprodutibilidade. `artifact_sha256` (bytes do arquivo)
e `output_pixel_sha256` (conteudo dos pixels) sao gravados **separados**.

In [ ]:
#@title 10. Gravar o recipe { display-mode: "form" }
import json, hashlib, pathlib, shutil
import numpy as np
from PIL import Image

RUN_DIR = pathlib.Path(
    f"/content/ChibiCreate/experiments/model_eval/{MODEL_KEY}/run_{RUN_ID}")
if RUN_DIR.exists():
    n = 2
    while (alt := RUN_DIR.parent / f"run_{RUN_ID}_r{n}").exists():
        n += 1
    RUN_DIR = alt
    print(f"[no overwrite] run anterior preservada -> {RUN_DIR.name}")
RUN_DIR.mkdir(parents=True)

destino = RUN_DIR / "output.png"
shutil.copy2(OUT, destino)
img = Image.open(destino)

RECIPE = {
    "benchmark": "wai_illustrious_sdxl_vs_flux2_klein_4b",
    "purpose": "comparative_benchmark_only",
    "is_official_pipeline": False,
    "run": RUN_ID,
    "character": CHARACTER_ID,

    "model": {
        "key": MODEL_KEY, "file": VERSAO["file"],
        "sha256": CKPT_SHA256,
        "civitai_model_id": VERSAO["civitai_model_id"],
        "civitai_model_version_id": VERSAO["civitai_model_version_id"],
        "license_displayed": VERSAO["license_displayed"],
        "license_verified_by_agent": False,
    },

    "reference_mechanism": "ipadapter_encode_combine",
    "reference_mechanism_note": " ".join(
        CFG["reference_equivalence_note"].split()),
    "ipadapter": IPADAPTER_META,

    "references": {
        papel: {
            "file": ENTRADAS[papel]["file"],
            "artifact_sha256": ENTRADAS[papel]["artifact_sha256"],
            "pixel_sha256": ENTRADAS[papel]["pixel_sha256"],
            "weight": SUBST[f"%%WEIGHT_{papel.upper()}%%"],
        } for papel in REFS_DESTA_RUN
    },
    "references_used": REFS_DESTA_RUN,
    "references_not_used": [p for p in ENTRADAS if p not in REFS_DESTA_RUN],
    "reference_count": len(REFS_DESTA_RUN),
    "weights_status": "BASELINE_EXPERIMENTAL",
    "combine_method": COMBINE_METHOD if IS_RUN_003 else None,

    "workflow": f"{WORKFLOW}@{WORKFLOW_VERSION}",
    "workflow_sha256": WORKFLOW_SHA256,
    "comfyui_commit": COMFY_COMMIT,

    "prompt": PROMPT,
    "negative_prompt": NEGATIVE,
    "parameters": {
        "seed": int(SEED), "steps": PAR["steps"], "cfg": PAR["cfg"],
        "sampler": PAR["sampler"], "scheduler": PAR["scheduler"],
        "denoise": PAR["denoise"], "resolution": PAR["resolution"],
        "batch": 1, "weight_type": "linear", "embeds_scaling": "V only",
    },

    "hardware": json.load(open("/content/gpu_info_wai.json")),
    "execution_time_s": EXEC_TIME,
    "artifact_sha256": hashlib.sha256(destino.read_bytes()).hexdigest(),
    "output_pixel_sha256": hashlib.sha256(
        np.array(img.convert("RGBA")).tobytes()).hexdigest(),
    "output_size": list(img.size),
    "determinism_note": (
        "Hash identico entre runs NAO e garantido. Nao prometemos "
        "determinismo absoluto entre execucoes."),
}
(RUN_DIR / "recipe.json").write_text(json.dumps(RECIPE, indent=2, default=str))

print("run dir :", RUN_DIR)
for k in ("artifact_sha256", "output_pixel_sha256", "workflow_sha256",
          "execution_time_s", "reference_count"):
    print(f"  {k:22} {RECIPE[k]}")
print(f"  {'pesos':22} {[(p, v['weight']) for p, v in RECIPE['references'].items()]}")


---

## Celula 11 — reprodutibilidade (001 vs 002)

Rode depois de ter executado as Runs 001 e 002.

In [ ]:
#@title 11. Comparar Run 001 x Run 002 { display-mode: "form" }
import json, pathlib

base = pathlib.Path(f"/content/ChibiCreate/experiments/model_eval/{MODEL_KEY}")
runs = {p.name: json.load(open(p / "recipe.json"))
        for p in sorted(base.glob("run_*")) if (p / "recipe.json").exists()}
print("runs registradas:", list(runs))

a, b = runs.get("run_001"), runs.get("run_002")
if not (a and b):
    print()
    print("Execute as Runs 001 e 002 antes desta comparacao.")
else:
    print()
    print("Entradas que DEVEM ser identicas:")
    for campo in ("prompt", "negative_prompt", "parameters",
                  "references_used", "workflow_sha256"):
        igual = a[campo] == b[campo]
        print(f"  {'OK   ' if igual else 'DIFERE'} {campo}")
        if not igual:
            print("    001:", a[campo])
            print("    002:", b[campo])
    assert a["model"]["sha256"] == b["model"]["sha256"], "checkpoint diferente"

    print()
    print("Saidas:")
    for campo in ("artifact_sha256", "output_pixel_sha256"):
        va, vb = a.get(campo), b.get(campo)
        print(f"  {campo:22} {'IDENTICO' if va == vb else 'DIFERENTE'}")
    print()
    print("Hash diferente NAO e falha: e o resultado da medicao.")
    print("Nao prometemos determinismo absoluto entre execucoes.")


---

## Celula 12 — montagem comparativa

Sem ranking automatico. A leitura e humana.

In [ ]:
#@title 12. ORIGINAL / FLUX 001 / FLUX 003 / WAI 001 / WAI 003 { display-mode: "form" }
import pathlib, json
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

ROOT = pathlib.Path("/content/ChibiCreate")

def _abrir(p):
    p = pathlib.Path(p) if p else None
    return Image.open(p).convert("RGB") if p and p.exists() else None

paineis = [(_abrir(ROOT / f"characters/{CHARACTER_ID}/reference/full_body.png"),
            "ORIGINAL")]

_fx = sorted((ROOT / "experiments/model_eval").rglob("*flux*/run_001/output.png"))
paineis.append((_abrir(_fx[0]) if _fx else None, "FLUX RUN 001\n(1 ref)"))

_fx3 = ROOT / f"characters/{CHARACTER_ID}/chibi/run_003_output.png"
paineis.append((_abrir(_fx3), "FLUX RUN 003\n(3 refs, ReferenceLatent)"))

_wai = ROOT / f"experiments/model_eval/{MODEL_KEY}"
paineis.append((_abrir(_wai / "run_001/output.png"), "WAI RUN 001\n(1 ref, IP-Adapter)"))
paineis.append((_abrir(_wai / "run_003/output.png"), "WAI RUN 003\n(3 refs, IP-Adapter)"))

paineis = [(im, t) for im, t in paineis if im is not None]
n = len(paineis)
fig, ax = plt.subplots(1, n, figsize=(4.6 * n, 6.5))
for a, (im, t) in zip(np.atleast_1d(ax), paineis):
    a.imshow(im); a.set_title(t, fontsize=10); a.axis("off")
plt.tight_layout()
plt.savefig("/content/wai_vs_flux.png", dpi=110, bbox_inches="tight")
plt.show()

print("=" * 62)
print("[HUMAN REVIEW REQUIRED] — nao ha ranking automatico")
print("=" * 62)
print("Eixos: STYLE / IDENTITY / DESIGN_PRESERVATION")
print("Eixo PRINCIPAL: DESIGN_PRESERVATION")
print()
for item in ["roupa", "capa", "ornamentos dourados", "acessorios", "cabelo",
             "chifres", "silhueta", "proporcoes chibi", "fidelidade das cores"]:
    print("  [ ]", item)
print()
print("Pergunta:")
print('  "O WAI produz uma roupa MAIS FIEL ao design original que o FLUX?"')
print("  SIM -> WAI vira candidato a ROTA DIRETA")
print("  NAO -> FLUX segue como baseline")
print()
print("MECANISMOS DIFERENTES, nao equivalentes internamente:")
print("  FLUX = multi-referencia via mecanismo proprio (ReferenceLatent)")
print("  WAI  = multi-referencia via IP-Adapter (embeds CLIP-Vision)")
print("A comparacao e sobre o RESULTADO VISUAL com as mesmas referencias.")


---

## PARE AQUI

Concluido o previsto: Runs 001, 002 e 003 com receitas e hashes registrados.

**Nao** implementar a rota `FLUX RUN 003 -> WAI` ainda. Primeiro medimos
`ORIGINAL -> WAI` contra `ORIGINAL -> FLUX`. So **se** o WAI mostrar vantagem
real em DESIGN_PRESERVATION e que essa rota se discute.

### [HUMAN REVIEW REQUIRED]

- Fixar a versao do checkpoint e ler a licenca da versao especifica.
- Avaliar os tres eixos e concluir com **uma** marca:
  `PROMISING` / `INSUFFICIENT` / `BLOCKED`.
- Se os pesos baseline (1.0 / 0.6 / 0.8) parecerem desequilibrados, isso e
  assunto de uma **rodada de calibracao** propria — nao deste benchmark.

Nada de Design Transfer, TPS, LoRA, ControlNet, sweep de prompt ou seed,
Pony, outro checkpoint WAI, animacao, master ou pipeline oficial.
